In [1]:
# !pip install -q transformers accelerate bitsandbytes sentence-transformers

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import os
from google.colab import drive

In [3]:
drive.mount('/content/drive')

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 1. Configuration for 4-bit loading (to fit on T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
def run_baseline_inference(model_id="deepseek-ai/deepseek-coder-7b-instruct-v1.5"):
    print(f"--- Loading Model: {model_id} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # Load gold set
    df = pd.read_csv('/content/drive/MyDrive/Project/data/gold_set.csv')
    results = []

    print("--- Starting Baseline Inference ---")
    for index, row in tqdm(df.iterrows(), total=len(df)):
        code = row['func_before']
        cwe = row.get('CWE ID', 'Unknown')

        # Baseline Prompt (Standard)
        prompt = f"Identify the vulnerability in this C code and explain why it is dangerous:\n\n{code}\n\nExplanation:"
        
        # Tokenization with Truncation (for large code snippets)
        inputs = tokenizer(
            prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=3072 
        ).to("cuda")
        
        # Generate output
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=256,
                temperature=0.2, # Low temperature for more deterministic/stable results
                do_sample=True
            )
        
        explanation = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        results.append({
            'index': index,
            'cwe': cwe,
            'code': code,
            'baseline_explanation': explanation.strip()
        })

    # Save results for comparison
    output_df = pd.DataFrame(results)
    
    os.makedirs('/content/drive/MyDrive/Project/results', exist_ok=True)
    output_df.to_csv('/content/drive/MyDrive/Project/results/baseline_results.csv', index=False)
    print("--- Baseline results saved to results/baseline_results.csv ---")


In [6]:
if __name__ == "__main__":
    run_baseline_inference()

--- Loading Model: deepseek-ai/deepseek-coder-7b-instruct-v1.5 ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

--- Starting Baseline Inference ---


100%|██████████| 50/50 [22:34<00:00, 27.08s/it]

--- Baseline results saved to results/baseline_results.csv ---
